# Práctica 3 — Expresión Diferencial con PyDESeq2 y PCA
## Análisis del knockdown de **UPF1** en hESCs (GSE81214)

**Máster en Bioinformática — Módulo de Transcriptómica**

Esta práctica continúa donde terminó la [Práctica 2](../02-rnaseq-upf1/): partiendo de la
matriz de conteos de **featureCounts** (4 muestras: 2 control vs 2 siRNA-UPF1), realizaremos:

1. **Exploración de la matriz de conteos** — estadísticas y visualizaciones
2. **Análisis de expresión diferencial** con **PyDESeq2** (implementación Python de DESeq2)
3. **Transformación VST y PCA** para evaluar la separación entre condiciones
4. **Validación biológica** — ¿se redujo la expresión de UPF1 en el knockdown?

> **Recordatorio:** trabajamos con la anotación reducida de 6 genes del chr19
> (UPF1, EML2, ITPKC, JUND, LPAR2, SIRT2) y **2 réplicas por condición**. Los resultados
> son ilustrativos del flujo de trabajo; con tan pocas réplicas la estimación de la
> dispersión es limitada (lo discutiremos en las preguntas).

## 0. Instalación

`pydeseq2` se instala desde PyPI junto con sus dependencias (anndata, scikit-learn…).

In [ ]:
# ============================================================================
# CELDA 0: Instalación (una vez por sesión)
# ============================================================================
!pip install -q pydeseq2 "pandas==2.2.3"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import pydeseq2
print(f"✓ pydeseq2 {pydeseq2.__version__} instalado")

## 1. Datos: matriz de conteos de featureCounts

Usaremos el archivo `gene_counts.txt` generado en la Práctica 2. Por defecto se descarga
la versión precomputada del repositorio del curso (`MODO = "URL"`), pero si tienes tu
propio resultado de la Práctica 2, cambia a `MODO = "SUBIR"` y súbelo.

In [ ]:
# ============================================================================
# CELDA 1: Obtención de la matriz de conteos
# ============================================================================
import urllib.request

MODO = "URL"   # "URL" | "SUBIR"

if MODO == "URL":
    DATA_DIR = Path("/content/datos_pydeseq2")
    DATA_DIR.mkdir(exist_ok=True)
    URL_BASE = "https://raw.githubusercontent.com/victorsanchezarevalo/transcriptomica-master/main/practicas/03-rnaseq-pydeseq2/datos/"
    counts_path = DATA_DIR / "gene_counts.txt"
    if not counts_path.exists():
        urllib.request.urlretrieve(URL_BASE + "gene_counts.txt", counts_path)

elif MODO == "SUBIR":
    from google.colab import files
    DATA_DIR = Path("/content")
    print("Sube tu archivo gene_counts.txt (de la Práctica 2):")
    files.upload()
    counts_path = DATA_DIR / "gene_counts.txt"

# Directorio de salida
OUT_DIR = Path("/content/pydeseq2_results")
OUT_DIR.mkdir(exist_ok=True)

# --- Carga del archivo featureCounts ---
# La primera línea es el comando ejecutado; la segunda, las cabeceras
counts_df = pd.read_csv(counts_path, sep="\t", skiprows=1)

# Renombrar columnas de muestras: quedarnos con el ID SRR
counts_df.columns = [c.split("/")[-1].split("_")[0] if "SRR" in c else c
                     for c in counts_df.columns]
counts_df = counts_df.set_index("Geneid")

# Nos quedamos solo con las columnas de conteos, como enteros
SAMPLES = ["SRR3480371", "SRR3480372", "SRR3480373", "SRR3480374"]
counts = counts_df[SAMPLES].astype(int)

# Diseño experimental
CONDICION = {"SRR3480371": "control", "SRR3480372": "control",
             "SRR3480373": "siRNA_UPF1", "SRR3480374": "siRNA_UPF1"}

print(f"✓ Matriz de conteos: {counts.shape[0]} genes × {counts.shape[1]} muestras")
print(f"  Conteos totales por muestra: {counts.sum().to_dict()}")
display(counts)

---
# PARTE I — Exploración de la matriz de conteos

Antes de cualquier análisis estadístico hay que **mirar los datos**: distribución de
conteos, rango dinámico y similitud entre muestras.

In [ ]:
# ============================================================================
# CELDA 2: Estadísticas descriptivas
# ============================================================================

print("Estadísticas por muestra:")
display(counts.describe().round(1))

counts_stats = pd.DataFrame({
    "mean_counts": counts.mean(axis=1),
    "total_counts": counts.sum(axis=1),
})
print("\nEstadísticas por gen:")
display(counts_stats.describe().round(1))

In [ ]:
# ============================================================================
# CELDA 3: Visualización de la distribución de conteos
# ============================================================================

plt.style.use('default')
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Histograma de conteos totales por gen
axes[0,0].hist(counts_stats['total_counts'], bins=50, alpha=0.7, color='skyblue', edgecolor='black')
axes[0,0].set_xlabel('Conteos totales por gen')
axes[0,0].set_ylabel('Frecuencia')
axes[0,0].set_title('Distribución de conteos totales por gen')
axes[0,0].set_yscale('log')

# 2. Boxplot de conteos por muestra
counts.boxplot(ax=axes[0,1])
axes[0,1].set_title('Distribución de conteos por muestra')
axes[0,1].set_ylabel('Conteos')
axes[0,1].tick_params(axis='x', rotation=45)

# 3. Distribución en escala log
log_counts = np.log10(counts_stats['total_counts'] + 1)
axes[1,0].hist(log_counts, bins=50, alpha=0.7, color='lightcoral', edgecolor='black')
axes[1,0].set_xlabel('Log10(conteos totales + 1)')
axes[1,0].set_ylabel('Frecuencia')
axes[1,0].set_title('Distribución en escala logarítmica')

# 4. Violin plot
counts_melted = counts.melt(var_name='Sample', value_name='Counts')
sns.violinplot(data=counts_melted, x='Sample', y='Counts', ax=axes[1,1])
axes[1,1].set_title('Distribución de conteos por muestra (Violin plot)')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(OUT_DIR / "exploracion_conteos.png", dpi=150)
plt.show()

> ❓ **Preguntas 1:**
> 1. ¿Cómo es la distribución de conteos por gen? ¿Por qué se usa la escala logarítmica
>    para visualizarla?
> 2. ¿Observas diferencias de profundidad de secuenciación (*library size*) entre muestras?
>    ¿Cómo las corrige DESeq2 internamente?

---
# PARTE II — Análisis de expresión diferencial con PyDESeq2

**PyDESeq2** reproduce en Python el método DESeq2 (Love et al., 2014): modela los conteos
con una **binomial negativa**, estima factores de tamaño y dispersiones, y aplica un test
de Wald sobre los coeficientes del modelo. Flujo:

1. `DeseqDataSet` — objeto con conteos + metadatos + diseño (`~condition`)
2. `dds.deseq2()` — estima tamaños de librería, dispersiones y ajusta el modelo
3. `DeseqStats` — test estadístico para el contraste siRNA_UPF1 vs control

In [ ]:
# ============================================================================
# CELDA 4: Metadatos y objeto DeseqDataSet
# ============================================================================
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

# Metadatos: una fila por muestra con su condición
metadata = pd.DataFrame({"condition": CONDICION})
metadata.index.name = "sample"
display(metadata)

inference = DefaultInference(n_cpus=2)

# NOTA: PyDESeq2 espera muestras en FILAS y genes en COLUMNAS → transponemos
dds = DeseqDataSet(
    counts=counts.T,
    metadata=metadata,
    design="~condition",
    refit_cooks=True,
    inference=inference,
)
print("✓ DeseqDataSet creado:", dds.n_obs, "muestras ×", dds.n_vars, "genes")

In [ ]:
# ============================================================================
# CELDA 5: Ejecutar el pipeline DESeq2
# ============================================================================

# Estima factores de tamaño, dispersiones y ajusta el modelo binomial negativo
dds.deseq2()

# Factores de tamaño estimados (normalización por profundidad de secuenciación)
print("Factores de tamaño por muestra:")
print(pd.Series(dds.obs["size_factors"], index=dds.obs_names).round(3))

In [ ]:
# ============================================================================
# CELDA 6: Test estadístico del contraste siRNA_UPF1 vs control
# ============================================================================

# Contraste: log2FC positivo = mayor expresión en siRNA_UPF1 que en control
stat = DeseqStats(dds, contrast=["condition", "siRNA_UPF1", "control"],
                  inference=inference)
stat.summary()

# Tabla de resultados: baseMean, log2FoldChange, pvalue, padj (BH)
results = stat.results_df.copy()
results.index.name = "Geneid"
display(results.round(4))

results.to_csv(OUT_DIR / "resultados_deseq2.csv")

In [ ]:
# ============================================================================
# CELDA 7: Clasificación y volcano plot
# ============================================================================

PADJ_UMBRAL = 0.05
LFC_UMBRAL = 1.0   # |log2FC| > 1 (cambio ×2)

def clasificar(row):
    if pd.isna(row["padj"]) or row["padj"] >= PADJ_UMBRAL:
        return "No significativo"
    return "Sobre-expresado" if row["log2FoldChange"] > LFC_UMBRAL else \
           ("Infra-expresado" if row["log2FoldChange"] < -LFC_UMBRAL else "No significativo")

results["Categoria"] = results.apply(clasificar, axis=1)
print(results["Categoria"].value_counts().to_string())

paleta = {"No significativo": "gray", "Sobre-expresado": "red", "Infra-expresado": "blue"}
fig, ax = plt.subplots(figsize=(9, 7))
for cat, color in paleta.items():
    sub = results[results["Categoria"] == cat]
    ax.scatter(sub["log2FoldChange"], -np.log10(sub["padj"]),
               c=color, alpha=0.7, s=40, label=f"{cat} (n={len(sub)})")
    for gen, row in sub.iterrows():
        ax.annotate(gen, (row["log2FoldChange"], -np.log10(row["padj"])),
                    xytext=(6, 4), textcoords="offset points", fontsize=8)

ax.axhline(-np.log10(PADJ_UMBRAL), ls="--", c="black", alpha=0.6)
ax.axvline(LFC_UMBRAL, ls="--", c="black", alpha=0.6)
ax.axvline(-LFC_UMBRAL, ls="--", c="black", alpha=0.6)
ax.set_xlabel("log2 Fold Change (siRNA_UPF1 vs control)")
ax.set_ylabel("-log10(padj)")
ax.set_title("Volcano plot — knockdown de UPF1 en hESCs")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "volcano_pydeseq2.png", dpi=150)
plt.show()

In [ ]:
# ============================================================================
# CELDA 8: Validación biológica — ¿funcionó el knockdown de UPF1?
# ============================================================================

# Conteos normalizados por tamaño de librería
norm = counts.div(dds.obs["size_factors"], axis=1)

upf1 = norm.loc[[g for g in norm.index if "ENSG00000005007" in g or g == "UPF1"][0]] \
       if any("ENSG00000005007" in g or g == "UPF1" for g in norm.index) else None

if upf1 is not None:
    df_upf1 = pd.DataFrame({"muestra": upf1.index, "conteos_norm": upf1.values,
                            "condicion": [CONDICION[s] for s in upf1.index]})
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.barplot(data=df_upf1, x="condicion", y="conteos_norm", errorbar=None,
                palette={"control": "#4C9F70", "siRNA_UPF1": "#E07B39"}, ax=ax)
    sns.stripplot(data=df_upf1, x="condicion", y="conteos_norm",
                  color="black", size=9, ax=ax)
    ax.set_title("UPF1 — conteos normalizados por condición")
    ax.set_ylabel("Conteos normalizados")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "UPF1_conteos_normalizados.png", dpi=150)
    plt.show()

    media_ctrl = df_upf1.query("condicion == 'control'")["conteos_norm"].mean()
    media_kd = df_upf1.query("condicion == 'siRNA_UPF1'")["conteos_norm"].mean()
    print(f"Media control: {media_ctrl:.1f} | Media knockdown: {media_kd:.1f}")
    print(f"Reducción de UPF1 en el knockdown: {100*(1 - media_kd/media_ctrl):.0f}%")
else:
    print("No se encontró UPF1 en la matriz (revisa el identificador de gen).")

> ❓ **Preguntas 2:**
> 1. ¿Cuántos genes resultan significativos con padj < 0.05? ¿Y si usamos el p-valor sin
>    corregir? ¿Qué nos enseña esto sobre la corrección por múltiples tests?
> 2. ¿Se reduce la expresión de UPF1 en el knockdown? ¿Es coherente con el diseño
>    experimental? ¿Qué ocurriría si no lo estuviera?
> 3. Con solo **2 réplicas por condición**, ¿qué limitaciones tiene la estimación de la
>    dispersión? ¿Cómo las mitiga DESeq2 "pidiendo prestada" información entre genes?
> 4. ¿Qué efecto tiene sobre los resultados trabajar con solo 6 genes en lugar del
>    transcriptoma completo?

---
# PARTE III — Transformación VST y PCA

Para visualizar la similitud global entre muestras aplicamos la **VST** (*Variance
Stabilizing Transformation*), que homogeneiza la varianza a lo largo del rango de
expresión, y después un **PCA**.

In [ ]:
# ============================================================================
# CELDA 9: VST + PCA
# ============================================================================
from sklearn.decomposition import PCA

# Variance Stabilizing Transformation (para visualización, no para el test)
dds.vst(use_design=False, fit_type="mean")
vst_counts = dds.layers["vst_counts"]

# PCA sobre las muestras (filas)
pca = PCA(n_components=2)
pca_results = pca.fit_transform(vst_counts)

pca_df = pd.DataFrame(pca_results, columns=["PC1", "PC2"], index=dds.obs_names)
pca_df["condition"] = metadata["condition"].values
pca_df["sample"] = pca_df.index

sns.set_style("whitegrid")
plt.figure(figsize=(10, 7))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="condition", s=150,
                alpha=0.85, edgecolor="black", linewidth=0.5)
for _, row in pca_df.iterrows():
    plt.annotate(row["sample"], (row["PC1"], row["PC2"]),
                 xytext=(6, 6), textcoords="offset points", fontsize=9, alpha=0.75)
plt.title("PCA de las muestras (datos VST)", fontsize=15, fontweight="bold", pad=15)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} varianza)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} varianza)")
plt.legend(title="Condición")
plt.tight_layout()
plt.savefig(OUT_DIR / "PCA_vst.png", dpi=150)
plt.show()

print(f"Varianza explicada PC1: {pca.explained_variance_ratio_[0]:.2%}")
print(f"Varianza explicada PC2: {pca.explained_variance_ratio_[1]:.2%}")

> ❓ **Preguntas 3:**
> 1. ¿Se agrupan las réplicas de la misma condición en el PCA? ¿Qué porcentaje de la
>    varianza explica PC1?
> 2. ¿Por qué es necesario aplicar VST (y no usar los conteos crudos) antes del PCA?
> 3. Con 4 muestras y 6 genes, ¿qué valor interpretativo tiene este PCA? ¿Qué esperarías
>    ver con el transcriptoma completo?

---
## Exportación de resultados

In [ ]:
# ============================================================================
# CELDA 10: Comprimir y descargar resultados
# ============================================================================
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/pydeseq2_results", "zip", OUT_DIR)
print(f"✓ ZIP creado: {zip_path}")
files.download(zip_path)

---
## Recordatorio de entrega

1. **PDF** con las respuestas a todas las preguntas ❓ y la interpretación de los
   resultados (¿se valida el knockdown? ¿qué genes cambian y en qué dirección?).
2. **Este notebook ejecutado** de principio a fin, con salidas visibles y tus comentarios.